# OrthoVision — F3/F4 training on Colab (GPU)

Runs the full baseline ladder and the LoRA adaptation of the BiomedCLIP image
encoder on the frozen test fold, then evaluates the ADD §7 gates with bootstrap
confidence intervals.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough).


## 1. Clone + install


In [ ]:
import os
if not os.path.isdir('OrthoVision'):
    !git clone https://github.com/LuizCarlosGoulart/OrthoVision.git
%cd OrthoVision
!pip install -e ".[models]" -q

## 2. Environment check


In [ ]:
import torch
print('torch', torch.__version__)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)
if dev == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — set Runtime -> GPU. LoRA on CPU is very slow.')

## 3. Materialize the data

The manifests (train/val/test) are versioned, but the raw images are not.
This downloads DENTEX (~11.8 GB) and extracts the images to `data/raw/dentex/`
at the paths the manifests reference. Run once per session.


In [ ]:
!python scripts/ingest_dentex.py
!python scripts/validate_dentex.py

## 4. Run baselines + LoRA (captures per-image scores for bootstrap)

Bump `epochs` in `configs/experiment/lora.yaml` (or below) for stronger LoRA.


In [ ]:
import torch
from orthovision.config import load_config, resolve_path
from orthovision.preprocess.transform import PreprocessConfig
from orthovision.labels.schema import read_canonical, PATHOLOGY_KEYS
from orthovision.datamodule.labels import label_columns, label_matrix
from orthovision.models.backbone import load_backbone
from orthovision.models.zero_shot import zero_shot_scores
from orthovision.models.linear_probe import train_linear_probe, probe_scores
from orthovision.models.lora_train import train_lora_head, lora_scores
from orthovision.models.baselines import _summarize
from orthovision.eval.metrics import per_class_metrics

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pre = PreprocessConfig.from_config(load_config('preprocess'))
exp = load_config('experiment/lora')['experiment']; lc, tc = exp['lora'], exp['train']
class_prompts = load_config('experiment/base')['experiment']['class_prompts']

train = read_canonical(resolve_path('manifests/train.jsonl'))
test  = read_canonical(resolve_path('manifests/test.jsonl'))
train_paths = [str(resolve_path(r.image_path)) for r in train]
test_paths  = [str(resolve_path(r.image_path)) for r in test]
y_train = torch.tensor(label_matrix(train), dtype=torch.float32).to(device)
test_labels = label_columns(test)

scores = {}
# generic CLIP zero-shot
bb = load_backbone(load_config('model/clip_vitb16'), pre, device)
scores['zs_generic'] = zero_shot_scores(bb, class_prompts, bb.encode_images(test_paths)); del bb
# BiomedCLIP zero-shot + linear probe (frozen encoder)
bb = load_backbone(load_config('model/biomedclip'), pre, device)
test_feats = bb.encode_images(test_paths)
scores['zs_biomed'] = zero_shot_scores(bb, class_prompts, test_feats)
train_feats = bb.encode_images(train_paths)
probe = train_linear_probe(train_feats, y_train, seed=exp['seed'])
scores['probe'] = probe_scores(probe, test_feats, PATHOLOGY_KEYS); del bb, test_feats, train_feats
# BiomedCLIP + LoRA (fresh backbone so the encoder adapts from scratch)
bb = load_backbone(load_config('model/biomedclip'), pre, device)
head = train_lora_head(bb, train_paths, y_train, rank=lc['rank'], alpha=lc['alpha'],
    epochs=tc['epochs'], batch_size=tc['batch_size'], lr=tc['lr'],
    weight_decay=tc['weight_decay'], seed=exp['seed'], targets=lc['targets'])
scores['lora'] = lora_scores(bb, head, test_paths, keys=PATHOLOGY_KEYS); del bb

summaries = {k: _summarize(per_class_metrics(v, test_labels)) for k, v in scores.items()}
for name, s in summaries.items():
    print(f"{name:<11} macroAUC {s['macro_auc']:.3f}  local {s['local_macro_auc']:.3f}  global {s['global_macro_auc']:.3f}")

## 5. Gates (ADD §7) + bootstrap CIs


In [ ]:
from orthovision.eval.gates import check_gates
from orthovision.eval.bootstrap import bootstrap_auc_ci, paired_delta_ci

gates = check_gates(summaries['lora'], summaries['probe'], summaries['zs_biomed'], summaries['zs_generic'])
print('GATES:', gates)
print('OVERALL PASS:', gates['pass'])
print()
# Decisive gate: caries LoRA vs probe (paired bootstrap; CI excluding 0 => significant)
d = paired_delta_ci(scores['lora']['caries'], scores['probe']['caries'], test_labels['caries'], seed=exp['seed'])
print(f"caries  LoRA-probe delta AUC {d['mean']:+.3f}  CI[{d['lo']:+.3f}, {d['hi']:+.3f}]")
print()
print('per-class LoRA AUC with 95% CI:')
for k in PATHOLOGY_KEYS:
    ci = bootstrap_auc_ci(scores['lora'][k], test_labels[k], seed=exp['seed'])
    print(f"  {k:<18} {summaries['lora']['per_class'][k]['auc']:.3f}  CI[{ci['lo']:.3f}, {ci['hi']:.3f}]")

## 6. Save results


In [ ]:
import json, time, pathlib
out = pathlib.Path('experiments') / f"colab_{time.strftime('%Y%m%d_%H%M%S')}"
out.mkdir(parents=True, exist_ok=True)
payload = {'summaries': summaries, 'gates': gates,
           'config': {'rank': lc['rank'], 'alpha': lc['alpha'], 'epochs': tc['epochs'],
                      'batch_size': tc['batch_size'], 'lr': tc['lr'], 'seed': exp['seed']}}
(out / 'results.json').write_text(json.dumps(payload, indent=2))
print('saved ->', out / 'results.json')
# Optional: copy to Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r {out} /content/drive/MyDrive/

## Notes

- **Tuning knobs** (`configs/experiment/lora.yaml`): `rank`, `alpha`, `targets`,
  `epochs`, `lr`. Start with the defaults, then raise `epochs` (e.g. 15–30) and
  try `rank` 8→16 if caries AUC is still below the probe (0.552).
- **Decisive gate:** caries AUC must beat the F3 linear probe (0.552); the paired
  CI excluding 0 makes it significant.
- Results are deterministic given the seed; the test fold is the same frozen split
  as local (manifests are versioned).
